# Lab 1 — Outlier Exploration

**Day 06 · Anomaly Detection · Cisco AI/ML Training**

---

## Learning objectives

1. Load the credit card transaction dataset (**1,000** rows, **10** fraud).
2. Count **IQR outliers** in `amount` and `distance_from_home`.
3. Compare fraud vs legitimate transaction profiles.
4. Explain why rule-based outlier flags alone are insufficient for fraud.

> **Checkpoints:** **1000** rows · **10** fraud · mean fraud amount ≈ **234** · legit ≈ **43**

**Companion script:** `../scripts/lab01_outlier_exploration.py`

## Outliers vs fraud

Fraudulent transactions often look like **outliers**, but not every outlier is fraud.

| Step | Tool |
|------|------|
| IQR rule | Flag values outside Q1 − 1.5×IQR to Q3 + 1.5×IQR |
| EDA | Compare means for `is_fraud` = 0 vs 1 |
| Later labs | ML models handle imbalance and mixed signals |

Day 5 found **noise** points in clustering; Day 6 hunts **rare fraud** in supervised/semi-supervised settings.

## Anomaly detection methods (course syllabus)

<!-- cisco-topic-coverage -->

| Method | Type | This course |
|--------|------|-------------|
| IQR / rules | Statistical | **This lab** |
| LOF | Proximity | Day 6 Lab 4 |
| Random Forest | Ensemble supervised | Day 6 Lab 5 |
| Isolation Forest | Ensemble unsupervised | Practice material |

### Prediction outliers
Models can flag transactions with **unusual predicted risk** or **large residuals** — combine rules (this lab) with ML scores (Labs 4–6).


---

## 1. Load transactions

In [ ]:
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif GH_ROOT.name == "day-06":
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "data" / "credit-card" / "credit_card_transactions.csv").is_file():
            GH_ROOT = parent
            break

df = pd.read_csv(GH_ROOT / "data" / "credit-card" / "credit_card_transactions.csv")

print("Lab 1 — Outlier exploration")
print(f"rows: {len(df)}")
print(f"fraud rows: {int(df['is_fraud'].sum())}")
print(f"fraud rate: {df['is_fraud'].mean():.4f}")
display(df.head(3))

---

## 2. IQR outlier helper

In [ ]:
def iqr_outlier_count(series: pd.Series) -> int:
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return int(((series < lower) | (series > upper)).sum())


def iqr_bounds(series: pd.Series) -> tuple[float, float]:
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr


amount_outliers = iqr_outlier_count(df["amount"])
distance_outliers = iqr_outlier_count(df["distance_from_home"])

print(f"IQR outliers (amount): {amount_outliers}")
print(f"IQR outliers (distance): {distance_outliers}")

**58** IQR flags on amount — but only **10** fraud rows total. Most outliers are legitimate high-spend transactions.

---

## 3. Fraud vs legitimate means

In [ ]:
legit_amount_mean = df.loc[df["is_fraud"] == 0, "amount"].mean()
fraud_amount_mean = df.loc[df["is_fraud"] == 1, "amount"].mean()
max_fraud_distance = df.loc[df["is_fraud"] == 1, "distance_from_home"].max()

print(f"mean amount (legit): {legit_amount_mean:.2f}")
print(f"mean amount (fraud): {fraud_amount_mean:.2f}")
print(f"max distance (fraud): {max_fraud_distance:.2f}")

compare = pd.DataFrame({
    "group": ["legit", "fraud"],
    "mean_amount": [legit_amount_mean, fraud_amount_mean],
    "count": [(df["is_fraud"] == 0).sum(), (df["is_fraud"] == 1).sum()],
})
display(compare.round(2))

---

## 4. Visualize amount and distance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

sns.boxplot(data=df, x="is_fraud", y="amount", ax=axes[0], palette="Set2")
axes[0].set_title("Transaction amount by fraud label")
axes[0].set_xticklabels(["legit (0)", "fraud (1)"])

sns.boxplot(data=df, x="is_fraud", y="distance_from_home", ax=axes[1], palette="Set2")
axes[1].set_title("Distance from home by fraud label")
axes[1].set_xticklabels(["legit (0)", "fraud (1)"])

plt.tight_layout()
plt.show()

---

## 5. Fraud transactions

In [ ]:
fraud_df = df.loc[df["is_fraud"] == 1].sort_values("amount", ascending=False)
display(fraud_df[["amount", "distance_from_home", "merchant_category"]].round(2))

---

## 6. Amounts above IQR upper bound

In [ ]:
lower, upper = iqr_bounds(df["amount"])
high_amount = df.loc[df["amount"] > upper, ["amount", "distance_from_home", "is_fraud", "merchant_category"]]

print(f"IQR upper bound (amount): {upper:.2f}")
print(f"rows above upper bound: {len(high_amount)}")
print(f"fraud among high-amount rows: {int(high_amount['is_fraud'].sum())}")
display(high_amount.head(8).round(2))

Rule-based flags create many **false positives** — motivation for Labs 2–6.

---

## 7. Checkpoint summary

In [ ]:
assert len(df) == 1000
assert int(df["is_fraud"].sum()) == 10
assert amount_outliers == 58
assert distance_outliers == 58
assert abs(legit_amount_mean - 43.22) < 1.0
assert abs(fraud_amount_mean - 233.94) < 5.0
assert abs(max_fraud_distance - 51.72) < 1.0
print("✓ All checkpoint assertions passed")

---

## Reflection questions

1. Why are there more IQR outliers than fraud cases?
2. Which features might help beyond amount and distance?
3. How does extreme imbalance (10 fraud) affect evaluation? *(Lab 2)*

**Previous:** [Day 05 — Unsupervised Learning](../day-05/README.md)  
**Next:** [Lab 2 — Imbalance analysis](lab02_imbalance_analysis.ipynb)